# FinFlow 1.0 — Phase 3: Mathematical Baseline Models
**Before running:** `Runtime → Change runtime type → T4 GPU`

This notebook trains all three baseline models and crowns the Phase 4 **Draft Pick**.

| Step | What happens | Est. time |
|------|-------------|----------|
| 1. Setup | Install libs, mount Drive | 1 min |
| 2. Upload | Push aligned CSVs to Colab | 1–2 min |
| 3. Preprocessing | Build 24h tensors | 1 min |
| 4. LSTM | Train + evaluate | 3–5 min |
| 5. Transformer | Train + evaluate | 4–6 min |
| 6. Prophet | Statistical baseline | 3–5 min |
| 7. Report | Comparison table + Draft Pick | instant |

In [ ]:
!pip install prophet scikit-learn -q

import torch, warnings
warnings.filterwarnings('ignore')
import logging; logging.getLogger('cmdstanpy').setLevel(logging.WARNING)
import logging; logging.getLogger('prophet').setLevel(logging.WARNING)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
if device.type == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
else:
    print('WARNING: No GPU! Go to Runtime → Change runtime type → T4 GPU')

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_OUT = '/content/drive/MyDrive/FinFlow/phase3_outputs'
os.makedirs(DRIVE_OUT, exist_ok=True)
os.makedirs('/content/models', exist_ok=True)
print(f'Outputs will save to: {DRIVE_OUT}')

In [ ]:
#Upload the two aligned CSVs

from google.colab import files
print('Upload: train_aligned.csv AND test_aligned.csv')
uploaded = files.upload()
for fname, data in uploaded.items():
    print(f'  ✓ {fname}  ({len(data)/1024/1024:.1f} MB)')

In [ ]:

# Full Preprocessing (scaler + 24h tensors)

import pandas as pd
import numpy as np
import pickle
from sklearn.preprocessing import StandardScaler

LOOKBACK = 24
TARGET_TICKERS = ['NVDA', 'TSLA', 'JPM', 'SPY']
VAL_SPLIT_DATE = pd.Timestamp('2024-09-01', tz='UTC')

FEATURE_COLS = [
    'returns', 'log_returns',
    'volatility_5h', 'volatility_24h',
    'volume_change', 'volume_ratio',
    'momentum_4h', 'momentum_12h', 'momentum_24h',
    'rsi_14',
    'macd', 'macd_signal', 'macd_hist',
    'bb_width', 'bb_pct',
    'vwap_spread', 'body_size', 'upper_wick', 'lower_wick',
]
N_FEATURES = len(FEATURE_COLS)
LABEL_MAP = {-1: 0, 0: 1, 1: 2}  # SELL→0, HOLD→1, BUY→2

def load_csv(fname_options):
    for fname in fname_options:
        if os.path.exists(fname):
            df = pd.read_csv(fname, parse_dates=['timestamp'])
            if df['timestamp'].dt.tz is None:
                df['timestamp'] = df['timestamp'].dt.tz_localize('UTC')
            return df
    raise FileNotFoundError(f'None found: {fname_options}')

train_raw = load_csv(['train_sentiment.csv', 'train_aligned.csv'])
test_raw  = load_csv(['test_sentiment.csv',  'test_aligned.csv'])
train_raw['label'] = train_raw['target_direction'].map(LABEL_MAP)
test_raw['label']  = test_raw['target_direction'].map(LABEL_MAP)
print(f'Loaded: train={train_raw.shape}, test={test_raw.shape}')

scalers = {}
for sym in TARGET_TICKERS:
    sc = StandardScaler()
    sc.fit(train_raw[train_raw['symbol']==sym][FEATURE_COLS].dropna())
    scalers[sym] = sc

def build_sequences(df, scalers):
    all_X, all_y, all_meta = [], [], []
    for sym in TARGET_TICKERS:
        sd = df[df['symbol']==sym].sort_values('timestamp').copy().reset_index(drop=True)
        valid = sd[FEATURE_COLS+['label']].notna().all(axis=1)
        sd = sd[valid].reset_index(drop=True)
        feats = scalers[sym].transform(sd[FEATURE_COLS].values.astype(np.float32))
        labels = sd['label'].values.astype(np.int64)
        timestamps = sd['timestamp'].values
        for i in range(LOOKBACK-1, len(sd)):
            w = feats[i-LOOKBACK+1:i+1]
            if np.isnan(w).any(): continue
            all_X.append(w)
            all_y.append(labels[i])
            all_meta.append({'symbol': sym, 'timestamp': timestamps[i]})
    return np.array(all_X, dtype=np.float32), np.array(all_y, dtype=np.int64), pd.DataFrame(all_meta)

X_all, y_all, meta_all = build_sequences(train_raw, scalers)
X_test, y_test, meta_test = build_sequences(test_raw, scalers)

# Chronological train/val split
ts = pd.to_datetime(meta_all['timestamp'])
if ts.dt.tz is None: ts = ts.dt.tz_localize('UTC')
val_mask = ts >= VAL_SPLIT_DATE
X_train, y_train = X_all[~val_mask], y_all[~val_mask]
X_val,   y_val   = X_all[val_mask],  y_all[val_mask]

print(f'X_train: {X_train.shape}  X_val: {X_val.shape}  X_test: {X_test.shape}')
print(f'Features: {N_FEATURES}  Lookback: {LOOKBACK}h')
for name, y in [('Train', y_train), ('Val', y_val), ('Test', y_test)]:
    c = np.bincount(y, minlength=3)
    print(f'  {name}: SELL={c[0]:,}({c[0]/len(y):.1%}) HOLD={c[1]:,}({c[1]/len(y):.1%}) BUY={c[2]:,}({c[2]/len(y):.1%})')

In [ ]:

# Model Definitions (LSTM + Transformer)

import torch.nn as nn
import math

class LSTMClassifier(nn.Module):
    def __init__(self, input_size=19, hidden_size=128, num_layers=2, dropout=0.3, num_classes=3):
        super().__init__()
        self.lstm = nn.LSTM(input_size, hidden_size, num_layers,
                            batch_first=True, dropout=dropout if num_layers>1 else 0)
        self.norm = nn.LayerNorm(hidden_size)
        self.drop = nn.Dropout(dropout)
        self.head = nn.Sequential(
            nn.Linear(hidden_size, 64), nn.ReLU(), nn.Dropout(0.2),
            nn.Linear(64, num_classes)
        )
    def forward(self, x):
        out, _ = self.lstm(x)
        return self.head(self.drop(self.norm(out[:, -1, :])))
    def count_parameters(self):
        return sum(p.numel() for p in self.parameters() if p.requires_grad)

class SinusoidalPE(nn.Module):
    def __init__(self, d_model, max_len=100, dropout=0.1):
        super().__init__()
        self.dropout = nn.Dropout(p=dropout)
        pe = torch.zeros(max_len, d_model)
        pos = torch.arange(0, max_len).unsqueeze(1).float()
        div = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0)/d_model))
        pe[:, 0::2] = torch.sin(pos * div)
        pe[:, 1::2] = torch.cos(pos * div)
        self.register_buffer('pe', pe.unsqueeze(0))
    def forward(self, x):
        return self.dropout(x + self.pe[:, :x.size(1)])

class TransformerClassifier(nn.Module):
    def __init__(self, input_size=19, d_model=64, nhead=4, num_layers=2,
                 dim_feedforward=256, dropout=0.1, num_classes=3):
        super().__init__()
        self.proj   = nn.Linear(input_size, d_model)
        self.pe     = SinusoidalPE(d_model, dropout=dropout)
        enc_layer   = nn.TransformerEncoderLayer(d_model, nhead, dim_feedforward,
                                                  dropout, 'gelu', batch_first=True, norm_first=True)
        self.encoder = nn.TransformerEncoder(enc_layer, num_layers)
        self.norm    = nn.LayerNorm(d_model)
        self.head    = nn.Sequential(
            nn.Linear(d_model, 32), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(32, num_classes)
        )
    def forward(self, x):
        x = self.pe(self.proj(x))
        x = self.norm(self.encoder(x)).mean(dim=1)
        return self.head(x)
    def count_parameters(self):
        return sum(p.numel() for p in self.parameters() if p.requires_grad)

# Quick forward pass test
dummy = torch.randn(4, 24, N_FEATURES)
lstm_test = LSTMClassifier(N_FEATURES)
tf_test   = TransformerClassifier(N_FEATURES)
with torch.no_grad():
    print(f'LSTM output:        {lstm_test(dummy).shape}  ({lstm_test.count_parameters():,} params) ✓')
    print(f'Transformer output: {tf_test(dummy).shape}  ({tf_test.count_parameters():,} params) ✓')

In [ ]:

# Shared Training Loop

from torch.utils.data import TensorDataset, DataLoader
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
import time

def get_class_weights(y, n_classes=3):
    counts = np.bincount(y, minlength=n_classes).astype(float)
    return torch.FloatTensor(len(y) / (n_classes * counts))

@torch.no_grad()
def predict(model, X, batch_size=512):
    model.eval()
    loader = DataLoader(TensorDataset(torch.FloatTensor(X)), batch_size=batch_size)
    return np.concatenate([model(xb.to(device)).argmax(1).cpu().numpy() for (xb,) in loader])

def train_model(name, model, X_tr, y_tr, X_v, y_v,
                lr=1e-3, wd=1e-4, epochs=60, bs=256, patience=12):
    model = model.to(device)
    cw = get_class_weights(y_tr).to(device)
    criterion = nn.CrossEntropyLoss(weight=cw)
    opt = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=wd)
    sch = torch.optim.lr_scheduler.ReduceLROnPlateau(opt, 'min', 0.5, patience=5, min_lr=1e-6)

    Xt, yt = torch.FloatTensor(X_tr), torch.LongTensor(y_tr)
    Xv, yv = torch.FloatTensor(X_v),  torch.LongTensor(y_v)
    tr_loader = DataLoader(TensorDataset(Xt, yt), bs, shuffle=True)
    vl_loader = DataLoader(TensorDataset(Xv, yv), bs*2, shuffle=False)

    history = {'train_loss': [], 'val_loss': [], 'val_acc': []}
    best_loss, patience_cnt, best_state = float('inf'), 0, None
    t0 = time.time()

    for ep in range(1, epochs+1):
        # Train
        model.train()
        tl_sum, tn = 0, 0
        for Xb, yb in tr_loader:
            Xb, yb = Xb.to(device), yb.to(device)
            opt.zero_grad()
            loss = criterion(model(Xb), yb)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()
            tl_sum += loss.item()*len(yb); tn += len(yb)

        # Validate
        model.eval()
        vl_sum, vn, vc = 0, 0, 0
        with torch.no_grad():
            for Xb, yb in vl_loader:
                Xb, yb = Xb.to(device), yb.to(device)
                lo = model(Xb)
                vl_sum += criterion(lo, yb).item()*len(yb)
                vn += len(yb); vc += (lo.argmax(1)==yb).sum().item()

        tl, vl, va = tl_sum/tn, vl_sum/vn, vc/vn
        sch.step(vl)
        history['train_loss'].append(tl)
        history['val_loss'].append(vl)
        history['val_acc'].append(va)

        if vl < best_loss - 1e-5:
            best_loss = vl; patience_cnt = 0
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
        else:
            patience_cnt += 1

        if ep % 5 == 0:
            lr_now = opt.param_groups[0]['lr']
            print(f'  Ep {ep:3d}/{epochs} | TrLoss={tl:.4f} VlLoss={vl:.4f} ValAcc={va:.2%} LR={lr_now:.1e} [{time.time()-t0:.0f}s]')

        if patience_cnt >= patience:
            print(f'  Early stop at epoch {ep}'); break

    model.load_state_dict(best_state)
    torch.save(model.state_dict(), f'/content/models/{name.lower()}_best.pt')
    # Also save to Drive
    torch.save(model.state_dict(), f'{DRIVE_OUT}/{name.lower()}_best.pt')
    print(f'  ✓ Best val loss: {best_loss:.4f} | Saved to Drive')
    return model, history

print('Training functions defined ✓')

In [ ]:
#LSTM
print(f'  LSTM  ({LSTMClassifier(N_FEATURES).count_parameters():,} params)')

lstm = LSTMClassifier(input_size=N_FEATURES)
lstm, lstm_history = train_model(
    'LSTM', lstm, X_train, y_train, X_val, y_val,
    lr=1e-3, epochs=60, bs=256, patience=12
)
lstm_preds = predict(lstm, X_test)
print('\n--- LSTM Test Set Report ---')
print(classification_report(y_test, lstm_preds, target_names=['SELL(0)', 'HOLD(1)', 'BUY(2)']))

In [ ]:
#Train Transformer

print(f'  Transformer  ({TransformerClassifier(N_FEATURES).count_parameters():,} params)')

transformer = TransformerClassifier(input_size=N_FEATURES)
transformer, tf_history = train_model(
    'Transformer', transformer, X_train, y_train, X_val, y_val,
    lr=5e-4, epochs=60, bs=256, patience=12
)
tf_preds = predict(transformer, X_test)
print('\n--- Transformer Test Set Report ---')
print(classification_report(y_test, tf_preds, target_names=['SELL(0)', 'HOLD(1)', 'BUY(2)']))

In [ ]:
# Prophet Baseline (per ticker)

from prophet import Prophet

THRESHOLD = 0.001
prophet_preds = np.ones(len(y_test), dtype=np.int64)  # default HOLD

def strip_tz(df, col):
    if df[col].dt.tz is not None:
        df[col] = df[col].dt.tz_localize(None)
    return df

tr_raw = strip_tz(train_raw.copy(), 'timestamp')
te_raw = strip_tz(test_raw.copy(), 'timestamp')
meta_t = strip_tz(meta_test.copy(), 'timestamp')

for sym in TARGET_TICKERS:
    print(f'{sym}: fitting...', end=' ', flush=True)
    sym_tr = tr_raw[tr_raw['symbol']==sym][['timestamp','close']].rename(columns={'timestamp':'ds','close':'y'}).dropna()
    sym_te = te_raw[te_raw['symbol']==sym][['timestamp','close']].rename(columns={'timestamp':'ds','close':'y'}).dropna()

    m = Prophet(changepoint_prior_scale=0.05, seasonality_mode='multiplicative',
                daily_seasonality=True, weekly_seasonality=True,
                yearly_seasonality=False, uncertainty_samples=0)
    m.fit(sym_tr)

    future   = m.make_future_dataframe(periods=len(sym_te)+1, freq='h', include_history=False)
    forecast = m.predict(future).set_index('ds')['yhat']

    n_buy, n_sell = 0, 0
    sym_idx = meta_t[meta_t['symbol']==sym].index
    for idx in sym_idx:
        ts  = pd.Timestamp(meta_t.loc[idx, 'timestamp'])
        row = sym_te[sym_te['ds']==ts]
        if len(row) == 0: continue
        actual = row['y'].values[0]
        future_fc = forecast[forecast.index > ts]
        if len(future_fc) == 0: continue
        pred_ret = (future_fc.iloc[0] - actual) / actual
        if pred_ret > THRESHOLD:   prophet_preds[idx] = 2; n_buy += 1
        elif pred_ret < -THRESHOLD: prophet_preds[idx] = 0; n_sell += 1

    n_hold = (meta_t['symbol']==sym).sum() - n_buy - n_sell
    print(f'BUY={n_buy}  SELL={n_sell}  HOLD={n_hold}')

print('\n--- Prophet Test Set Report ---')
print(classification_report(y_test, prophet_preds, target_names=['SELL(0)', 'HOLD(1)', 'BUY(2)']))

In [ ]:
# Final Comparison Table + Draft Pick

from sklearn.metrics import classification_report, accuracy_score

def summarize(name, y_pred, y_true):
    r = classification_report(y_true, y_pred, target_names=['SELL(0)','HOLD(1)','BUY(2)'], output_dict=True)
    return {
        'model':          name,
        'accuracy':       round(accuracy_score(y_true, y_pred), 4),
        'buy_precision':  round(r.get('BUY(2)',{}).get('precision', 0), 4),
        'sell_precision': round(r.get('SELL(0)',{}).get('precision', 0), 4),
        'buy_f1':         round(r.get('BUY(2)',{}).get('f1-score', 0), 4),
        'weighted_f1':    round(r.get('weighted avg',{}).get('f1-score', 0), 4),
    }

results = [
    summarize('LSTM',        lstm_preds,    y_test),
    summarize('Transformer', tf_preds,      y_test),
    summarize('Prophet',     prophet_preds, y_test),
]

results_df = pd.DataFrame(results)
print('\n' + '='*70)
print('  PHASE 3 FINAL COMPARISON TABLE — 2025 UNSEEN TEST SET')
print(results_df.to_string(index=False))

# Draft pick = highest (buy_precision + sell_precision + weighted_f1) / 3
results_df['score'] = (results_df['buy_precision'] + results_df['sell_precision'] + results_df['weighted_f1']) / 3
draft_pick = results_df.loc[results_df['score'].idxmax(), 'model']

print(f'\n  🏆  DRAFT PICK: {draft_pick}')
print(f'      Carries into Phase 4 Late-Fusion NLP Override Layer')
print()
print('  FinBERT Emergency Brake rule (Phase 4):')
print(f'  If {draft_pick} predicts BUY/SELL AND sentiment_score ≤ -0.50 → veto to HOLD')
print()
print('  Academic benchmark: >38% acc on 3-class = beating random.')
print('  52-54% is excellent for raw price data.  >60% = check for overfitting.')

# Save results
results_df.to_csv(f'{DRIVE_OUT}/phase3_results.csv', index=False)
print(f'\n  Results saved → {DRIVE_OUT}/phase3_results.csv')

In [ ]:
import matplotlib.pyplot as plt
import matplotlib

BG, PANEL, BORDER = '#0a0a0f', '#0f1117', '#1e2330'
TEXT, MUTED = '#c8cdd6', '#5a6070'
ORANGE, BLUE, RED, GREEN = '#f97316', '#3b82f6', '#ef4444', '#22c55e'

fig, axes = plt.subplots(2, 2, figsize=(14, 9))
fig.patch.set_facecolor(BG)
fig.suptitle('PHASE 3  |  Training Curves  |  LSTM vs Transformer',
             color=TEXT, fontsize=13, fontweight='bold', x=0.02, ha='left')

hists = {'LSTM': lstm_history, 'Transformer': tf_history}
colors = {'LSTM': BLUE, 'Transformer': ORANGE}

for ax in axes.flat:
    ax.set_facecolor(PANEL); ax.tick_params(colors=MUTED, labelsize=9)
    for sp in ax.spines.values(): sp.set_edgecolor(BORDER)
    ax.grid(True, color=BORDER, lw=0.5, alpha=0.7, ls='--'); ax.set_axisbelow(True)

for i, (mname, hist) in enumerate(hists.items()):
    c = colors[mname]
    eps = range(1, len(hist['train_loss'])+1)

    # Loss
    axes[0,i].plot(eps, hist['train_loss'], color=c, lw=1.5, label='Train Loss')
    axes[0,i].plot(eps, hist['val_loss'], color=RED, lw=1.5, ls='--', label='Val Loss')
    best_ep = int(np.argmin(hist['val_loss']))+1
    axes[0,i].axvline(best_ep, color=GREEN, lw=1, ls=':', label=f'Best ep={best_ep}')
    axes[0,i].set_title(f'{mname} — Loss', color=c, fontweight='bold', fontsize=10)
    axes[0,i].legend(facecolor=PANEL, edgecolor=BORDER, labelcolor=TEXT, fontsize=8)

    # Accuracy
    va_pct = [v*100 for v in hist['val_acc']]
    axes[1,i].plot(eps, va_pct, color=c, lw=1.5)
    axes[1,i].axhline(52, color=MUTED, lw=0.8, ls=':', label='52% (profitable threshold)')
    axes[1,i].axhline(60, color=RED,   lw=0.8, ls=':', label='60% (overfit alert)')
    axes[1,i].set_title(f'{mname} — Val Accuracy', color=c, fontweight='bold', fontsize=10)
    axes[1,i].set_ylabel('Accuracy %', color=MUTED, fontsize=9)
    axes[1,i].legend(facecolor=PANEL, edgecolor=BORDER, labelcolor=TEXT, fontsize=8)

plt.tight_layout()
out_path = f'{DRIVE_OUT}/phase3_training_curves.png'
plt.savefig(out_path, dpi=150, bbox_inches='tight', facecolor=BG)
plt.show()
print(f'Saved → {out_path}')

In [ ]:
#Confusion Matrices 

from sklearn.metrics import confusion_matrix
from matplotlib.colors import LinearSegmentedColormap

model_preds = [('LSTM', lstm_preds, BLUE), ('Transformer', tf_preds, ORANGE), ('Prophet', prophet_preds, '#22c55e')]
label_names = ['SELL', 'HOLD', 'BUY']

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.patch.set_facecolor(BG)
fig.suptitle('PHASE 3  |  Confusion Matrices — 2025 Unseen Test Set',
             color=TEXT, fontsize=13, fontweight='bold', x=0.02, ha='left')

for ax, (mname, preds, color) in zip(axes, model_preds):
    cm = confusion_matrix(y_test, preds)
    cm_n = cm.astype(float) / cm.sum(axis=1, keepdims=True)
    cmap = LinearSegmentedColormap.from_list('bb', [PANEL, color], N=256)

    ax.set_facecolor(PANEL)
    im = ax.imshow(cm_n, cmap=cmap, vmin=0, vmax=1)
    fig.colorbar(im, ax=ax, fraction=0.04).ax.tick_params(colors=MUTED, labelsize=7)

    ax.set_xticks(range(3)); ax.set_yticks(range(3))
    ax.set_xticklabels(label_names, color=TEXT, fontweight='bold')
    ax.set_yticklabels(label_names, color=TEXT, fontweight='bold')
    ax.set_xlabel('Predicted', color=TEXT); ax.set_ylabel('Actual', color=TEXT)

    res = next(r for r in results if r['model']==mname)
    ax.set_title(f"{mname}\nAcc={res['accuracy']:.2%}  BUY-P={res['buy_precision']:.3f}",
                 color=color, fontweight='bold', pad=8)

    for i in range(3):
        for j in range(3):
            ax.text(j, i, f"{cm[i,j]:,}\n({cm_n[i,j]:.1%})",
                    ha='center', va='center', fontsize=9,
                    color='white' if cm_n[i,j] < 0.6 else 'black',
                    fontweight='bold' if i==j else 'normal')

plt.tight_layout()
out_path = f'{DRIVE_OUT}/phase3_confusion_matrices.png'
plt.savefig(out_path, dpi=150, bbox_inches='tight', facecolor=BG)
plt.show()
print(f'Saved → {out_path}')
print(f'\n✓ Phase 3 complete!')
print(f'Download from Drive: {DRIVE_OUT}/')
print('Files needed for Phase 4: [model]_best.pt  +  phase3_results.csv')